# Imports

In [1]:
import os
import scanpy as sc
import anndata as ad
import numpy as np
from scipy.stats import median_abs_deviation
from rpy2.robjects.conversion import localconverter
from rpy2.robjects import numpy2ri, pandas2ri
from scipy.sparse import csc_matrix
import matplotlib.pyplot as plt


In [2]:
def is_outlier(adata, metric: str, nmads: int):
    M = adata.obs[metric]
    print(f"median for {metric}", np.median(M))
    outlier = (M < np.median(M) - nmads * median_abs_deviation(M)) | (
        np.median(M) + nmads * median_abs_deviation(M) < M
    )
    return outlier

In [3]:
def calculate_and_plot_qc(adata_path, sample, adata_dict):
    adata = sc.read_10x_h5(adata_path)
    adata.var_names_make_unique()
    # mitochondrial genes, "MT-" for human, "Mt-" for mouse
    adata.var["mt"] = adata.var_names.str.startswith("mt-")
    # ribosomal genes
    adata.var["ribo"] = adata.var_names.str.startswith(("Rps", "Rpl"))
    # hemoglobin genes
    adata.var["hb"] = adata.var_names.str.startswith(("Hba-","Hbb-"))
    # calculate QC metrics
    sc.pp.calculate_qc_metrics(adata, qc_vars=["mt", "ribo", "hb"], percent_top=(20, 50, 100, 200, 500), inplace=True, log1p=True)
    # plotting of QC metrics
    g =  sc.pl.violin(
        adata,
        ["n_genes_by_counts", "total_counts", "pct_counts_mt", "pct_counts_ribo", "pct_counts_hb"],
        jitter=0.4,
        multi_panel=True,
        show = False
    )
    g.fig.suptitle(f"QC of {sample}",  y=1.05)
    

    ax = sc.pl.violin(
        adata,
        "total_counts",
        jitter=0.4,
        show=False
    )
    ax.set_ylim(0, 30000)
    ax.set_title(f"total counts zoom of {sample}")
    plt.show()

    ax = sc.pl.scatter(adata, "total_counts", "n_genes_by_counts", color="pct_counts_mt", show = False)
    ax.set_title(f"QC of {sample} mito as color")
    plt.show()

    ax = sc.pl.scatter(adata, "total_counts", "n_genes_by_counts", color="pct_counts_ribo", show = False)
    ax.set_title(f"QC of {sample} ribo as color")
    plt.show()
    adata_dict[sample] = adata
    return adata_dict

In [ ]:
samples = {
    "S26_17": "/projects/circ_iri/work/cell_ranger/output/S26_17/outs/filtered_feature_bc_matrix.h5",
    "S26_18": "/projects/circ_iri/work/cell_ranger/output/S26_18/outs/filtered_feature_bc_matrix.h5",
    "S26_19": "/projects/circ_iri/work/cell_ranger/output/S26_19/outs/filtered_feature_bc_matrix.h5",
    "S26_20": "/projects/circ_iri/work/cell_ranger/output/S26_20/outs/filtered_feature_bc_matrix.h5",
    "S26_22": "/projects/circ_iri/work/cell_ranger/output/S26_22/outs/filtered_feature_bc_matrix.h5",
    "S26_23": "/projects/circ_iri/work/cell_ranger/output/S26_23/outs/filtered_feature_bc_matrix.h5",
    "S26_27": "/projects/circ_iri/work/cell_ranger/output/S26_27/outs/filtered_feature_bc_matrix.h5",
    "S26_31": "/projects/circ_iri/work/cell_ranger/output/S26_31/outs/filtered_feature_bc_matrix.h5",
    "S26_32": "/projects/circ_iri/work/cell_ranger/output/S26_32/outs/filtered_feature_bc_matrix.h5",
    }

In [5]:
%%capture
adata_dict = {}
for sample, adata_path in samples.items():
    print(sample)
    adata_dict = calculate_and_plot_qc(adata_path, sample, adata_dict)

In [6]:
adata_dict

{'S26_17': AnnData object with n_obs × n_vars = 16435 × 33696
     obs: 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'pct_counts_in_top_20_genes', 'pct_counts_in_top_50_genes', 'pct_counts_in_top_100_genes', 'pct_counts_in_top_200_genes', 'pct_counts_in_top_500_genes', 'total_counts_mt', 'log1p_total_counts_mt', 'pct_counts_mt', 'total_counts_ribo', 'log1p_total_counts_ribo', 'pct_counts_ribo', 'total_counts_hb', 'log1p_total_counts_hb', 'pct_counts_hb'
     var: 'gene_ids', 'feature_types', 'genome', 'mt', 'ribo', 'hb', 'n_cells_by_counts', 'mean_counts', 'log1p_mean_counts', 'pct_dropout_by_counts', 'total_counts', 'log1p_total_counts',
 'S26_18': AnnData object with n_obs × n_vars = 18133 × 33696
     obs: 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'pct_counts_in_top_20_genes', 'pct_counts_in_top_50_genes', 'pct_counts_in_top_100_genes', 'pct_counts_in_top_200_genes', 'pct_counts_in_top_500_genes', '

In [7]:
def filter_adata(adata, mad, mito_mad):
    adata.obs["outlier"] = (
        is_outlier(adata, "log1p_total_counts", mad)
        | is_outlier(adata, "log1p_n_genes_by_counts", mad)
        | is_outlier(adata, "pct_counts_in_top_20_genes", mad)
    )
    adata.obs.outlier.value_counts()

    adata.obs["mt_outlier"] = is_outlier(adata, "pct_counts_mt", mito_mad) | (
    adata.obs["pct_counts_mt"] > 8
    )
    adata.obs.mt_outlier.value_counts()

    print(f"Total number of cells: {adata.n_obs}")
    adata_fil1 = adata[(~adata.obs.outlier) & (~adata.obs.mt_outlier)].copy()

    print(f"Number of cells after filtering of low quality cells: {adata_fil1.n_obs}")
    # p1 = sc.pl.scatter(adata_fil1, "total_counts", "n_genes_by_counts", color="pct_counts_mt")

    print(f"Number of cells after filtering of low quality cells: {adata_fil1.n_obs}")
    # p1 = sc.pl.scatter(adata_fil1, "total_counts", "n_genes_by_counts", color="pct_counts_ribo")
    return adata_fil1
    
def try_different_mads(adata, sample):
    print(sample)
    mad = 5
    mito_mad = 3
    print(f"Filter with MAD {mad} and mito_mad {mito_mad}")
    filter_adata(adata, mad, mito_mad)
    
    mad = 4
    print(f"Filter with MAD {mad} and mito_mad {mito_mad}")
    filter_adata(adata, mad, mito_mad)
    
    mad = 3
    print(f"Filter with MAD {mad} and mito_mad {mito_mad}")
    filter_adata(adata, mad, mito_mad)


In [8]:
# for sample, adata in adata_dict.items():
#     try_different_mads(adata, sample)

Filtering criteria set to MAD of 5 and MAD for mito gene


In [9]:
outdir = "/projects/circ_iri/work/sc_analysis/qc_plots"
os.makedirs(outdir, exist_ok=True)

In [10]:
mad_counts = 5
mito_mad = 3

filtered_adata_dict = {}

for sample, adata in adata_dict.items():

    filtered_adata = filter_adata(adata, mad_counts, mito_mad)
    filtered_adata_dict[sample] = filtered_adata

    # -------- QC multi-panel plot --------
    g = sc.pl.violin(
        filtered_adata,
        ["n_genes_by_counts", "total_counts", "pct_counts_mt", "pct_counts_ribo"],
        jitter=0.4,
        multi_panel=True,
        show=False
    )

    g.fig.suptitle(
        f"QC of {sample} (MAD_counts={mad_counts}, mito_mad={mito_mad})",
        y=1.05
    )

    qc_plot_path = os.path.join(
        outdir,
        f"{sample}_QC_violin_MADcounts{mad_counts}_MADmito{mito_mad}.png"
    )

    g.fig.savefig(qc_plot_path, dpi=300, bbox_inches="tight")
    plt.close(g.fig)


    # -------- total_counts zoom --------
    ax = sc.pl.violin(
        filtered_adata,
        "total_counts",
        jitter=0.4,
        show=False
    )

    ax.set_ylim(0, 10000)
    ax.set_title(
        f"total_counts zoom of {sample} (MAD_counts={mad_counts})"
    )

    total_counts_path = os.path.join(
        outdir,
        f"{sample}_total_counts_zoom_MADcounts{mad_counts}.png"
    )

    ax.figure.savefig(total_counts_path, dpi=300, bbox_inches="tight")
    plt.close(ax.figure)


    # -------- n_genes_by_counts zoom --------
    ax = sc.pl.violin(
        filtered_adata,
        "n_genes_by_counts",
        jitter=0.4,
        show=False
    )

    ax.set_ylim(0, 10000)
    ax.set_title(
        f"n_genes_by_counts zoom of {sample} (mito_mad={mito_mad})"
    )

    genes_path = os.path.join(
        outdir,
        f"{sample}_n_genes_zoom_MADgenes{mito_mad}.png"
    )

    ax.figure.savefig(genes_path, dpi=300, bbox_inches="tight")
    plt.close(ax.figure)

median for log1p_total_counts 8.02617
median for log1p_n_genes_by_counts 7.4770384723196965
median for pct_counts_in_top_20_genes 14.429530201342283
median for pct_counts_mt 1.1687144
Total number of cells: 16435
Number of cells after filtering of low quality cells: 15223
Number of cells after filtering of low quality cells: 15223
median for log1p_total_counts 8.033983
median for log1p_n_genes_by_counts 7.4949862339505335
median for pct_counts_in_top_20_genes 12.504624491305957
median for pct_counts_mt 1.1808368
Total number of cells: 18133
Number of cells after filtering of low quality cells: 16725
Number of cells after filtering of low quality cells: 16725
median for log1p_total_counts 7.77191
median for log1p_n_genes_by_counts 7.34826567703814
median for pct_counts_in_top_20_genes 10.857607194147192
median for pct_counts_mt 1.3917973
Total number of cells: 19450
Number of cells after filtering of low quality cells: 17200
Number of cells after filtering of low quality cells: 17200
me

# Ambient RNA check

In [11]:
import logging
import rpy2.rinterface_lib.callbacks as rcb
import rpy2.robjects as ro
from rpy2.robjects import pandas2ri

rcb.logger.setLevel(logging.ERROR)


%load_ext rpy2.ipython

In [12]:
%%R
library(SoupX)
library(Seurat)
library(scater)
library(scDblFinder)
library(SingleCellExperiment)
library(BiocParallel)

Loading required package: SeuratObject
Loading required package: sp

Attaching package: ‘SeuratObject’

The following objects are masked from ‘package:base’:

    intersect, t

Loading required package: SingleCellExperiment
Loading required package: SummarizedExperiment
Loading required package: MatrixGenerics
Loading required package: matrixStats

Attaching package: ‘MatrixGenerics’

The following objects are masked from ‘package:matrixStats’:

    colAlls, colAnyNAs, colAnys, colAvgsPerRowSet, colCollapse,
    colCounts, colCummaxs, colCummins, colCumprods, colCumsums,
    colDiffs, colIQRDiffs, colIQRs, colLogSumExps, colMadDiffs,
    colMads, colMaxs, colMeans2, colMedians, colMins, colOrderStats,
    colProds, colQuantiles, colRanges, colRanks, colSdDiffs, colSds,
    colSums2, colTabulates, colVarDiffs, colVars, colWeightedMads,
    colWeightedMeans, colWeightedMedians, colWeightedSds,
    colWeightedVars, rowAlls, rowAnyNAs, rowAnys, rowAvgsPerColSet,
    rowCollapse, rowCounts,

In [13]:
samples_raw_dict = {
    "S26_17": "/projects/circ_iri/work/cell_ranger/output/S26_17/outs/raw_feature_bc_matrix.h5",
    "S26_18": "/projects/circ_iri/work/cell_ranger/output/S26_18/outs/raw_feature_bc_matrix.h5",
    "S26_19": "/projects/circ_iri/work/cell_ranger/output/S26_19/outs/raw_feature_bc_matrix.h5",
    "S26_20": "/projects/circ_iri/work/cell_ranger/output/S26_20/outs/raw_feature_bc_matrix.h5",
    "S26_22": "/projects/circ_iri/work/cell_ranger/output/S26_22/outs/raw_feature_bc_matrix.h5",
    "S26_23": "/projects/circ_iri/work/cell_ranger/output/S26_23/outs/raw_feature_bc_matrix.h5",
    "S26_27": "/projects/circ_iri/work/cell_ranger/output/S26_27/outs/raw_feature_bc_matrix.h5",
    "S26_31": "/projects/circ_iri/work/cell_ranger/output/S26_31/outs/raw_feature_bc_matrix.h5",
    "S26_32": "/projects/circ_iri/work/cell_ranger/output/S26_32/outs/raw_feature_bc_matrix.h5",
    }

In [14]:
def prepare_adata_for_soup(adata, raw_path):
    adata_pp = adata.copy()
    sc.pp.normalize_total(adata_pp, target_sum=1e4)
    sc.pp.log1p(adata_pp)
    sc.pp.pca(adata_pp)
    sc.pp.neighbors(adata_pp)
    sc.tl.leiden(
        adata_pp, key_added="soupx_groups", flavor="igraph", n_iterations=2, directed=False
    )

    # Preprocess variables for SoupX
    adata.obs["soupx_groups"] = adata_pp.obs["soupx_groups"]
    del adata_pp
    cells = adata.obs_names
    genes = adata.var_names
    data = adata.X.T
    adata_raw = sc.read_10x_h5(raw_path)
    adata_raw.var_names_make_unique()

    genes_raw = adata_raw.var_names
    cells_raw = adata_raw.obs_names

    data_tod = adata_raw.X.T
    del adata_raw
    data_csc = data.tocsc()
    data_tod_csc = data_tod.tocsc()

    # Extract sparse components and cast to correct types
    x = data_csc.data.astype(np.float64)
    i = data_csc.indices.astype(np.int32)
    p = data_csc.indptr.astype(np.int32)
    dims = np.array(data_csc.shape, dtype=np.int32)

    x_tod = data_tod_csc.data.astype(np.float64)
    i_tod = data_tod_csc.indices.astype(np.int32)
    p_tod = data_tod_csc.indptr.astype(np.int32)
    dims_tod = np.array(data_tod_csc.shape, dtype=np.int32)

    with localconverter(ro.default_converter + pandas2ri.converter + numpy2ri.converter):
        ro.globalenv["x"] = x
        ro.globalenv["i"] = i
        ro.globalenv["p"] = p
        ro.globalenv["dims"] = dims

        ro.globalenv["x_tod"] = x_tod
        ro.globalenv["i_tod"] = i_tod
        ro.globalenv["p_tod"] = p_tod
        ro.globalenv["dims_tod"] = dims_tod

        ro.globalenv["genes"] = np.array(genes)
        ro.globalenv["genes_raw"] = np.array(genes_raw)
        ro.globalenv["cells"] = np.array(cells)
        ro.globalenv["cells_raw"] = np.array(cells_raw)
        ro.globalenv["soupx_groups"] = adata.obs["soupx_groups"].to_numpy()
    return data_csc, data_tod_csc, adata

In [15]:
%%R

library(Matrix)

prepare_soup_data_R <- function(){
        # Manually coerce types to avoid "array" class errors
        x <- as.numeric(x)
        i <- as.integer(i)
        p <- as.integer(p)
        dims <- as.integer(dims)

        x_tod <- as.numeric(x_tod)
        i_tod <- as.integer(i_tod)
        p_tod <- as.integer(p_tod)
        dims_tod <- as.integer(dims_tod)

        # Reconstruct sparse matrices
        data <- new("dgCMatrix",
                    Dim = dims,
                    x = x,
                    i = i,
                    p = p)

        data_tod <- new("dgCMatrix",
                        Dim = dims_tod,
                        x = x_tod,
                        i = i_tod,
                        p = p_tod)

        # Assign row and column names
        rownames(data) <- genes
        colnames(data) <- cells
        rownames(data_tod) <- genes_raw
        colnames(data_tod) <- cells_raw

        # SoupX pipeline
        sc = SoupChannel(data_tod, data, calcSoupProfile = TRUE)
        sc = setClusters(sc, soupx_groups)
        sc = autoEstCont(sc, doPlot = FALSE)
        out = adjustCounts(sc, roundToInt = TRUE)
        return(out)
}


Attaching package: ‘Matrix’

The following object is masked from ‘package:S4Vectors’:

    expand



In [16]:
def convert_soup_output_to_scanpy(adata):
    with localconverter(ro.default_converter + pandas2ri.converter + numpy2ri.converter):
        out_py = ro.conversion.rpy2py(ro.globalenv["out"])

    x = np.array(out_py.slots["x"])
    i = np.array(out_py.slots["i"])
    p = np.array(out_py.slots["p"])
    shape = tuple(out_py.slots["Dim"])

    out_matrix = csc_matrix((x, i, p), shape=shape)
    adata.layers["counts"] = adata.X.copy()
    adata.layers["soupX_counts"] = out_matrix.T
    adata.X = adata.layers["soupX_counts"]
    print(f"Total number of genes: {adata.n_vars}")

    # Min 20 cells - filters out 0 count genes
    sc.pp.filter_genes(adata, min_cells=20)
    print(f"Number of genes after cell filter: {adata.n_vars}")
    return adata

In [17]:
# only define dict once!!!
soup_adata_dict = {}

## S26_17

In [18]:
data_csc, data_tod_csc, adata = prepare_adata_for_soup(filtered_adata_dict["S26_17"], samples_raw_dict["S26_17"])

/home/aumlauf/miniforge3/envs/circ_iri/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/aumlauf/miniforge3/envs/circ_iri/lib/python3.11/site-packages/anndata/_core/anndata.py:1825: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
/home/aumlauf/miniforge3/envs/circ_iri/lib/python3.11/site-packages/anndata/_core/anndata.py:1825: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")


In [19]:
%%R -o out
out <- prepare_soup_data_R()

1036 genes passed tf-idf cut-off and 273 soup quantile filter.  Taking the top 100.
Using 890 independent estimates of rho.
Estimated global rho of 0.05
Expanding counts from 23 clusters to 15223 cells.
In addition: Warning message:
In sparseMatrix(i = out@i[w] + 1, j = out@j[w] + 1, x = out@x[w],  :
  'giveCsparse' is deprecated; setting repr="T" for you


In [20]:
soup_adata_dict["S26_17"] = convert_soup_output_to_scanpy(adata)

Total number of genes: 33696
Number of genes after cell filter: 18086


## S26_18

In [27]:
data_csc, data_tod_csc, adata = prepare_adata_for_soup(filtered_adata_dict["S26_18"], samples_raw_dict["S26_18"])

/home/aumlauf/miniforge3/envs/circ_iri/lib/python3.11/site-packages/anndata/_core/anndata.py:1825: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
/home/aumlauf/miniforge3/envs/circ_iri/lib/python3.11/site-packages/anndata/_core/anndata.py:1825: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")


In [28]:
%%R -o out
out <- prepare_soup_data_R()

2555 genes passed tf-idf cut-off and 405 soup quantile filter.  Taking the top 100.
Using 1130 independent estimates of rho.
Estimated global rho of 0.04
Expanding counts from 26 clusters to 16725 cells.
In addition: Warning message:
In sparseMatrix(i = out@i[w] + 1, j = out@j[w] + 1, x = out@x[w],  :
  'giveCsparse' is deprecated; setting repr="T" for you


In [29]:
soup_adata_dict["S26_18"] = convert_soup_output_to_scanpy(adata)

Total number of genes: 33696
Number of genes after cell filter: 18725


## S26_19

In [30]:
data_csc, data_tod_csc, adata = prepare_adata_for_soup(filtered_adata_dict["S26_19"], samples_raw_dict["S26_19"])

/home/aumlauf/miniforge3/envs/circ_iri/lib/python3.11/site-packages/anndata/_core/anndata.py:1825: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
/home/aumlauf/miniforge3/envs/circ_iri/lib/python3.11/site-packages/anndata/_core/anndata.py:1825: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")


In [31]:
%%R -o out
out <- prepare_soup_data_R()

2860 genes passed tf-idf cut-off and 560 soup quantile filter.  Taking the top 100.
Using 1340 independent estimates of rho.
Estimated global rho of 0.06
Expanding counts from 26 clusters to 17200 cells.
In addition: Warning message:
In sparseMatrix(i = out@i[w] + 1, j = out@j[w] + 1, x = out@x[w],  :
  'giveCsparse' is deprecated; setting repr="T" for you


In [32]:
soup_adata_dict["S26_19"] = convert_soup_output_to_scanpy(adata)

Total number of genes: 33696
Number of genes after cell filter: 18017


## S26_20

In [33]:
data_csc, data_tod_csc, adata = prepare_adata_for_soup(filtered_adata_dict["S26_20"], samples_raw_dict["S26_20"])

/home/aumlauf/miniforge3/envs/circ_iri/lib/python3.11/site-packages/anndata/_core/anndata.py:1825: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
/home/aumlauf/miniforge3/envs/circ_iri/lib/python3.11/site-packages/anndata/_core/anndata.py:1825: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")


In [34]:
%%R -o out
out <- prepare_soup_data_R()

4552 genes passed tf-idf cut-off and 842 soup quantile filter.  Taking the top 100.
Using 1587 independent estimates of rho.
Estimated global rho of 0.02
Expanding counts from 28 clusters to 15729 cells.
In addition: Warning message:
In sparseMatrix(i = out@i[w] + 1, j = out@j[w] + 1, x = out@x[w],  :
  'giveCsparse' is deprecated; setting repr="T" for you


In [35]:
soup_adata_dict["S26_20"] = convert_soup_output_to_scanpy(adata)

Total number of genes: 33696
Number of genes after cell filter: 18681


## S26_22

In [36]:
data_csc, data_tod_csc, adata = prepare_adata_for_soup(filtered_adata_dict["S26_22"], samples_raw_dict["S26_22"])

/home/aumlauf/miniforge3/envs/circ_iri/lib/python3.11/site-packages/anndata/_core/anndata.py:1825: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
/home/aumlauf/miniforge3/envs/circ_iri/lib/python3.11/site-packages/anndata/_core/anndata.py:1825: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")


In [37]:
%%R -o out
out <- prepare_soup_data_R()

2986 genes passed tf-idf cut-off and 499 soup quantile filter.  Taking the top 100.
Using 1450 independent estimates of rho.
Estimated global rho of 0.04
Expanding counts from 28 clusters to 12931 cells.
In addition: Warning message:
In sparseMatrix(i = out@i[w] + 1, j = out@j[w] + 1, x = out@x[w],  :
  'giveCsparse' is deprecated; setting repr="T" for you


In [38]:
soup_adata_dict["S26_22"] = convert_soup_output_to_scanpy(adata)

Total number of genes: 33696
Number of genes after cell filter: 19171


## S26_23

In [39]:
data_csc, data_tod_csc, adata = prepare_adata_for_soup(filtered_adata_dict["S26_23"], samples_raw_dict["S26_23"])

/home/aumlauf/miniforge3/envs/circ_iri/lib/python3.11/site-packages/anndata/_core/anndata.py:1825: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
/home/aumlauf/miniforge3/envs/circ_iri/lib/python3.11/site-packages/anndata/_core/anndata.py:1825: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")


In [40]:
%%R -o out
out <- prepare_soup_data_R()

4082 genes passed tf-idf cut-off and 551 soup quantile filter.  Taking the top 100.
Using 1085 independent estimates of rho.
Estimated global rho of 0.06
Expanding counts from 24 clusters to 12517 cells.
In addition: Warning message:
In sparseMatrix(i = out@i[w] + 1, j = out@j[w] + 1, x = out@x[w],  :
  'giveCsparse' is deprecated; setting repr="T" for you


In [41]:
soup_adata_dict["S26_23"] = convert_soup_output_to_scanpy(adata)

Total number of genes: 33696
Number of genes after cell filter: 17821


## S26_27

In [42]:
data_csc, data_tod_csc, adata = prepare_adata_for_soup(filtered_adata_dict["S26_27"], samples_raw_dict["S26_27"])

/home/aumlauf/miniforge3/envs/circ_iri/lib/python3.11/site-packages/anndata/_core/anndata.py:1825: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
/home/aumlauf/miniforge3/envs/circ_iri/lib/python3.11/site-packages/anndata/_core/anndata.py:1825: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")


In [43]:
%%R -o out
out <- prepare_soup_data_R()

1854 genes passed tf-idf cut-off and 489 soup quantile filter.  Taking the top 100.
Using 1616 independent estimates of rho.
Estimated global rho of 0.04
Expanding counts from 27 clusters to 14289 cells.
In addition: Warning message:
In sparseMatrix(i = out@i[w] + 1, j = out@j[w] + 1, x = out@x[w],  :
  'giveCsparse' is deprecated; setting repr="T" for you


In [44]:
soup_adata_dict["S26_27"] = convert_soup_output_to_scanpy(adata)

Total number of genes: 33696
Number of genes after cell filter: 18337


## S26_31


In [45]:
data_csc, data_tod_csc, adata = prepare_adata_for_soup(filtered_adata_dict["S26_31"], samples_raw_dict["S26_31"])

/home/aumlauf/miniforge3/envs/circ_iri/lib/python3.11/site-packages/anndata/_core/anndata.py:1825: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
/home/aumlauf/miniforge3/envs/circ_iri/lib/python3.11/site-packages/anndata/_core/anndata.py:1825: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")


In [46]:
%%R -o out
out <- prepare_soup_data_R()

2319 genes passed tf-idf cut-off and 496 soup quantile filter.  Taking the top 100.
Using 1375 independent estimates of rho.
Estimated global rho of 0.03
Expanding counts from 25 clusters to 12890 cells.
In addition: Warning message:
In sparseMatrix(i = out@i[w] + 1, j = out@j[w] + 1, x = out@x[w],  :
  'giveCsparse' is deprecated; setting repr="T" for you


In [47]:
soup_adata_dict["S26_31"] = convert_soup_output_to_scanpy(adata)

Total number of genes: 33696
Number of genes after cell filter: 18150


## S26_32

In [48]:
data_csc, data_tod_csc, adata = prepare_adata_for_soup(filtered_adata_dict["S26_32"], samples_raw_dict["S26_32"])

/home/aumlauf/miniforge3/envs/circ_iri/lib/python3.11/site-packages/anndata/_core/anndata.py:1825: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
/home/aumlauf/miniforge3/envs/circ_iri/lib/python3.11/site-packages/anndata/_core/anndata.py:1825: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")


In [49]:
%%R -o out
out <- prepare_soup_data_R()

2548 genes passed tf-idf cut-off and 394 soup quantile filter.  Taking the top 100.
Using 1334 independent estimates of rho.
Estimated global rho of 0.08
Expanding counts from 24 clusters to 9915 cells.
In addition: Warning message:
In sparseMatrix(i = out@i[w] + 1, j = out@j[w] + 1, x = out@x[w],  :
  'giveCsparse' is deprecated; setting repr="T" for you


In [50]:
soup_adata_dict["S26_32"] = convert_soup_output_to_scanpy(adata)

Total number of genes: 33696
Number of genes after cell filter: 17620


# Doublet Detection 

In [21]:
def prepare_adata_for_doublet_detection(adata):

    data_mat = adata.X.T.tocsc()

    x = data_mat.data.astype(np.float64)
    i = data_mat.indices.astype(np.int32)
    p = data_mat.indptr.astype(np.int32)
    dims = np.array(data_mat.shape, dtype=np.int32)

    with localconverter(ro.default_converter + numpy2ri.converter):
        ro.globalenv["x"] = x
        ro.globalenv["i"] = i
        ro.globalenv["p"] = p
        ro.globalenv["dims"] = dims

In [22]:
%%R 

get_doublet_score <- function(){
    x <- as.numeric(x)
    i <- as.integer(i)
    p <- as.integer(p)
    dims <- as.integer(dims)

    data_mat <- new("dgCMatrix", Dim = dims, x = x, i = i, p = p)

    set.seed(123)
    sce <- scDblFinder(SingleCellExperiment(list(counts = data_mat)))

    doublet_score <- sce$scDblFinder.score
    doublet_class <- sce$scDblFinder.class
    return(list(doublet_score = doublet_score, doublet_class = doublet_class))
}

In [23]:
def write_doublet_output_to_adata(adata, sample):
    adata.obs["scDblFinder_score"] = doublet_score
    adata.obs["scDblFinder_class"] = doublet_class
    adata.obs.scDblFinder_class.value_counts()
    adata.write(f"/projects/circ_iri/work/sc_analysis/{sample}_preprocessed.h5ad")
    return adata

## S26_17

In [24]:
prepare_adata_for_doublet_detection(soup_adata_dict["S26_17"])

In [25]:
%%R -o doublet_score -o doublet_class


result <- get_doublet_score()

doublet_score <- result$doublet_score
doublet_class <- result$doublet_class

Creating ~12179 artificial doublets...
Dimensional reduction
Evaluating kNN...
Training model...
iter=0, 2216 cells excluded from training.
iter=1, 2059 cells excluded from training.
iter=2, 2033 cells excluded from training.
Threshold found:0.485
1089 (7.2%) doublets called
In addition: There were 11 warnings (use warnings() to see them)


In [26]:
write_doublet_output_to_adata(soup_adata_dict["S26_17"], "S26_17")

AnnData object with n_obs × n_vars = 15223 × 18086
    obs: 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'pct_counts_in_top_20_genes', 'pct_counts_in_top_50_genes', 'pct_counts_in_top_100_genes', 'pct_counts_in_top_200_genes', 'pct_counts_in_top_500_genes', 'total_counts_mt', 'log1p_total_counts_mt', 'pct_counts_mt', 'total_counts_ribo', 'log1p_total_counts_ribo', 'pct_counts_ribo', 'total_counts_hb', 'log1p_total_counts_hb', 'pct_counts_hb', 'outlier', 'mt_outlier', 'soupx_groups', 'scDblFinder_score', 'scDblFinder_class'
    var: 'gene_ids', 'feature_types', 'genome', 'mt', 'ribo', 'hb', 'n_cells_by_counts', 'mean_counts', 'log1p_mean_counts', 'pct_dropout_by_counts', 'total_counts', 'log1p_total_counts', 'n_cells'
    layers: 'counts', 'soupX_counts'

## S26_18

In [51]:
prepare_adata_for_doublet_detection(soup_adata_dict["S26_18"])

In [52]:
%%R -o doublet_score -o doublet_class


result <- get_doublet_score()

doublet_score <- result$doublet_score
doublet_class <- result$doublet_class

Creating ~13380 artificial doublets...
Dimensional reduction
Evaluating kNN...
Training model...
iter=0, 2748 cells excluded from training.
iter=1, 2671 cells excluded from training.
iter=2, 2554 cells excluded from training.
Threshold found:0.48
1599 (9.6%) doublets called
In addition: There were 15 warnings (use warnings() to see them)


In [53]:
write_doublet_output_to_adata(soup_adata_dict["S26_18"], "S26_18")

AnnData object with n_obs × n_vars = 16725 × 18725
    obs: 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'pct_counts_in_top_20_genes', 'pct_counts_in_top_50_genes', 'pct_counts_in_top_100_genes', 'pct_counts_in_top_200_genes', 'pct_counts_in_top_500_genes', 'total_counts_mt', 'log1p_total_counts_mt', 'pct_counts_mt', 'total_counts_ribo', 'log1p_total_counts_ribo', 'pct_counts_ribo', 'total_counts_hb', 'log1p_total_counts_hb', 'pct_counts_hb', 'outlier', 'mt_outlier', 'soupx_groups', 'scDblFinder_score', 'scDblFinder_class'
    var: 'gene_ids', 'feature_types', 'genome', 'mt', 'ribo', 'hb', 'n_cells_by_counts', 'mean_counts', 'log1p_mean_counts', 'pct_dropout_by_counts', 'total_counts', 'log1p_total_counts', 'n_cells'
    layers: 'counts', 'soupX_counts'

## S26_19

In [54]:
prepare_adata_for_doublet_detection(soup_adata_dict["S26_19"])

In [55]:
%%R -o doublet_score -o doublet_class


result <- get_doublet_score()

doublet_score <- result$doublet_score
doublet_class <- result$doublet_class

Creating ~13760 artificial doublets...
Dimensional reduction
Evaluating kNN...
Training model...
iter=0, 2802 cells excluded from training.
iter=1, 2319 cells excluded from training.
iter=2, 2071 cells excluded from training.
Threshold found:0.375
1055 (6.1%) doublets called
In addition: There were 15 warnings (use warnings() to see them)


In [56]:
write_doublet_output_to_adata(soup_adata_dict["S26_19"], "S26_19")

AnnData object with n_obs × n_vars = 17200 × 18017
    obs: 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'pct_counts_in_top_20_genes', 'pct_counts_in_top_50_genes', 'pct_counts_in_top_100_genes', 'pct_counts_in_top_200_genes', 'pct_counts_in_top_500_genes', 'total_counts_mt', 'log1p_total_counts_mt', 'pct_counts_mt', 'total_counts_ribo', 'log1p_total_counts_ribo', 'pct_counts_ribo', 'total_counts_hb', 'log1p_total_counts_hb', 'pct_counts_hb', 'outlier', 'mt_outlier', 'soupx_groups', 'scDblFinder_score', 'scDblFinder_class'
    var: 'gene_ids', 'feature_types', 'genome', 'mt', 'ribo', 'hb', 'n_cells_by_counts', 'mean_counts', 'log1p_mean_counts', 'pct_dropout_by_counts', 'total_counts', 'log1p_total_counts', 'n_cells'
    layers: 'counts', 'soupX_counts'

## S26_20

In [57]:
prepare_adata_for_doublet_detection(soup_adata_dict["S26_20"])

In [58]:
%%R -o doublet_score -o doublet_class


result <- get_doublet_score()

doublet_score <- result$doublet_score
doublet_class <- result$doublet_class

Creating ~12584 artificial doublets...
Dimensional reduction
Evaluating kNN...
Training model...
iter=0, 2458 cells excluded from training.
iter=1, 2512 cells excluded from training.
iter=2, 2520 cells excluded from training.
Threshold found:0.464
1621 (10.3%) doublets called
In addition: There were 15 warnings (use warnings() to see them)


In [59]:
write_doublet_output_to_adata(soup_adata_dict["S26_20"], "S26_20")

AnnData object with n_obs × n_vars = 15729 × 18681
    obs: 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'pct_counts_in_top_20_genes', 'pct_counts_in_top_50_genes', 'pct_counts_in_top_100_genes', 'pct_counts_in_top_200_genes', 'pct_counts_in_top_500_genes', 'total_counts_mt', 'log1p_total_counts_mt', 'pct_counts_mt', 'total_counts_ribo', 'log1p_total_counts_ribo', 'pct_counts_ribo', 'total_counts_hb', 'log1p_total_counts_hb', 'pct_counts_hb', 'outlier', 'mt_outlier', 'soupx_groups', 'scDblFinder_score', 'scDblFinder_class'
    var: 'gene_ids', 'feature_types', 'genome', 'mt', 'ribo', 'hb', 'n_cells_by_counts', 'mean_counts', 'log1p_mean_counts', 'pct_dropout_by_counts', 'total_counts', 'log1p_total_counts', 'n_cells'
    layers: 'counts', 'soupX_counts'

## S26_22

In [60]:
prepare_adata_for_doublet_detection(soup_adata_dict["S26_22"])

In [61]:
%%R -o doublet_score -o doublet_class


result <- get_doublet_score()

doublet_score <- result$doublet_score
doublet_class <- result$doublet_class

Creating ~10345 artificial doublets...
Dimensional reduction
Evaluating kNN...
Training model...
iter=0, 2386 cells excluded from training.
iter=1, 2306 cells excluded from training.
iter=2, 2234 cells excluded from training.
Threshold found:0.479
1709 (13.2%) doublets called
In addition: There were 11 warnings (use warnings() to see them)


In [62]:
write_doublet_output_to_adata(soup_adata_dict["S26_22"], "S26_22")

AnnData object with n_obs × n_vars = 12931 × 19171
    obs: 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'pct_counts_in_top_20_genes', 'pct_counts_in_top_50_genes', 'pct_counts_in_top_100_genes', 'pct_counts_in_top_200_genes', 'pct_counts_in_top_500_genes', 'total_counts_mt', 'log1p_total_counts_mt', 'pct_counts_mt', 'total_counts_ribo', 'log1p_total_counts_ribo', 'pct_counts_ribo', 'total_counts_hb', 'log1p_total_counts_hb', 'pct_counts_hb', 'outlier', 'mt_outlier', 'soupx_groups', 'scDblFinder_score', 'scDblFinder_class'
    var: 'gene_ids', 'feature_types', 'genome', 'mt', 'ribo', 'hb', 'n_cells_by_counts', 'mean_counts', 'log1p_mean_counts', 'pct_dropout_by_counts', 'total_counts', 'log1p_total_counts', 'n_cells'
    layers: 'counts', 'soupX_counts'

## S26_23

In [63]:
prepare_adata_for_doublet_detection(soup_adata_dict["S26_23"])

In [64]:
%%R -o doublet_score -o doublet_class


result <- get_doublet_score()

doublet_score <- result$doublet_score
doublet_class <- result$doublet_class

Creating ~10014 artificial doublets...
Dimensional reduction
Evaluating kNN...
Training model...
iter=0, 2436 cells excluded from training.
iter=1, 2492 cells excluded from training.
iter=2, 2499 cells excluded from training.
Threshold found:0.48
1681 (13.4%) doublets called
In addition: There were 11 warnings (use warnings() to see them)


In [65]:
write_doublet_output_to_adata(soup_adata_dict["S26_23"], "S26_23")

AnnData object with n_obs × n_vars = 12517 × 17821
    obs: 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'pct_counts_in_top_20_genes', 'pct_counts_in_top_50_genes', 'pct_counts_in_top_100_genes', 'pct_counts_in_top_200_genes', 'pct_counts_in_top_500_genes', 'total_counts_mt', 'log1p_total_counts_mt', 'pct_counts_mt', 'total_counts_ribo', 'log1p_total_counts_ribo', 'pct_counts_ribo', 'total_counts_hb', 'log1p_total_counts_hb', 'pct_counts_hb', 'outlier', 'mt_outlier', 'soupx_groups', 'scDblFinder_score', 'scDblFinder_class'
    var: 'gene_ids', 'feature_types', 'genome', 'mt', 'ribo', 'hb', 'n_cells_by_counts', 'mean_counts', 'log1p_mean_counts', 'pct_dropout_by_counts', 'total_counts', 'log1p_total_counts', 'n_cells'
    layers: 'counts', 'soupX_counts'

## S26_27

In [66]:
prepare_adata_for_doublet_detection(soup_adata_dict["S26_27"])

In [67]:
%%R -o doublet_score -o doublet_class


result <- get_doublet_score()

doublet_score <- result$doublet_score
doublet_class <- result$doublet_class

Creating ~11432 artificial doublets...
Dimensional reduction
Evaluating kNN...
Training model...
iter=0, 2778 cells excluded from training.
iter=1, 2799 cells excluded from training.
iter=2, 2753 cells excluded from training.
Threshold found:0.525
1974 (13.8%) doublets called
In addition: There were 11 warnings (use warnings() to see them)


In [68]:
write_doublet_output_to_adata(soup_adata_dict["S26_27"], "S26_27")

AnnData object with n_obs × n_vars = 14289 × 18337
    obs: 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'pct_counts_in_top_20_genes', 'pct_counts_in_top_50_genes', 'pct_counts_in_top_100_genes', 'pct_counts_in_top_200_genes', 'pct_counts_in_top_500_genes', 'total_counts_mt', 'log1p_total_counts_mt', 'pct_counts_mt', 'total_counts_ribo', 'log1p_total_counts_ribo', 'pct_counts_ribo', 'total_counts_hb', 'log1p_total_counts_hb', 'pct_counts_hb', 'outlier', 'mt_outlier', 'soupx_groups', 'scDblFinder_score', 'scDblFinder_class'
    var: 'gene_ids', 'feature_types', 'genome', 'mt', 'ribo', 'hb', 'n_cells_by_counts', 'mean_counts', 'log1p_mean_counts', 'pct_dropout_by_counts', 'total_counts', 'log1p_total_counts', 'n_cells'
    layers: 'counts', 'soupX_counts'

## S26_31

In [69]:
prepare_adata_for_doublet_detection(soup_adata_dict["S26_31"])

In [70]:
%%R -o doublet_score -o doublet_class


result <- get_doublet_score()

doublet_score <- result$doublet_score
doublet_class <- result$doublet_class

Creating ~10312 artificial doublets...
Dimensional reduction
Evaluating kNN...
Training model...
iter=0, 2497 cells excluded from training.
iter=1, 2510 cells excluded from training.
iter=2, 2492 cells excluded from training.
Threshold found:0.489
1702 (13.2%) doublets called
In addition: There were 11 warnings (use warnings() to see them)


In [71]:
write_doublet_output_to_adata(soup_adata_dict["S26_31"], "S26_31")

AnnData object with n_obs × n_vars = 12890 × 18150
    obs: 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'pct_counts_in_top_20_genes', 'pct_counts_in_top_50_genes', 'pct_counts_in_top_100_genes', 'pct_counts_in_top_200_genes', 'pct_counts_in_top_500_genes', 'total_counts_mt', 'log1p_total_counts_mt', 'pct_counts_mt', 'total_counts_ribo', 'log1p_total_counts_ribo', 'pct_counts_ribo', 'total_counts_hb', 'log1p_total_counts_hb', 'pct_counts_hb', 'outlier', 'mt_outlier', 'soupx_groups', 'scDblFinder_score', 'scDblFinder_class'
    var: 'gene_ids', 'feature_types', 'genome', 'mt', 'ribo', 'hb', 'n_cells_by_counts', 'mean_counts', 'log1p_mean_counts', 'pct_dropout_by_counts', 'total_counts', 'log1p_total_counts', 'n_cells'
    layers: 'counts', 'soupX_counts'

## S26_32

In [72]:
prepare_adata_for_doublet_detection(soup_adata_dict["S26_32"])

In [73]:
%%R -o doublet_score -o doublet_class


result <- get_doublet_score()

doublet_score <- result$doublet_score
doublet_class <- result$doublet_class

Creating ~7932 artificial doublets...
Dimensional reduction
Evaluating kNN...
Training model...
iter=0, 1682 cells excluded from training.
iter=1, 1726 cells excluded from training.
iter=2, 1692 cells excluded from training.
Threshold found:0.505
1129 (11.4%) doublets called
In addition: Warning messages:
1: In .pause_resource() :
  omp_pause_resource() failed to relinquish resources on device 0
2: In .pause_resource() :
  omp_pause_resource() failed to relinquish resources on device 0
3: In .pause_resource() :
  omp_pause_resource() failed to relinquish resources on device 0
4: In .pause_resource() :
  omp_pause_resource() failed to relinquish resources on device 0
5: In .pause_resource() :
  omp_pause_resource() failed to relinquish resources on device 0
6: In .pause_resource() :
  omp_pause_resource() failed to relinquish resources on device 0
7: In .pause_resource() :
  omp_pause_resource() failed to relinquish resources on device 0


In [74]:
write_doublet_output_to_adata(soup_adata_dict["S26_32"], "S26_32")

AnnData object with n_obs × n_vars = 9915 × 17620
    obs: 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'pct_counts_in_top_20_genes', 'pct_counts_in_top_50_genes', 'pct_counts_in_top_100_genes', 'pct_counts_in_top_200_genes', 'pct_counts_in_top_500_genes', 'total_counts_mt', 'log1p_total_counts_mt', 'pct_counts_mt', 'total_counts_ribo', 'log1p_total_counts_ribo', 'pct_counts_ribo', 'total_counts_hb', 'log1p_total_counts_hb', 'pct_counts_hb', 'outlier', 'mt_outlier', 'soupx_groups', 'scDblFinder_score', 'scDblFinder_class'
    var: 'gene_ids', 'feature_types', 'genome', 'mt', 'ribo', 'hb', 'n_cells_by_counts', 'mean_counts', 'log1p_mean_counts', 'pct_dropout_by_counts', 'total_counts', 'log1p_total_counts', 'n_cells'
    layers: 'counts', 'soupX_counts'